In [2]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [84]:
# Sex by age bands
# To download original dataset go to - 
#https://www.nomisweb.co.uk/query/construct/components/simpleapicomponent.aspx?menuopt=10930&subcomp=
student_acc_by_age_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Students\LC4411EW - Student accommodation by age - Census 2011.csv"
student_pop_2021_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Students\lsoa2021_2019_lookup_table_with_student_total.csv"

## 2. Input Link to LSOA 2011 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2011-boundaries-ew-bfe-v3/about

In [4]:
lsoa2011_shapefile = r"N:\Geodatabase\Raw_Data\Census 2011\Boundaries\Lower_Layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Extent_BFE_EW_V3_2022_4117212475924899368(1)\LSOA_2011_EW_BFE_V3.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [5]:
threshold_value = 2.5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [6]:
def add_percentage_columns(df, total_col, suffix='_count', new_suffix='_perc', columns_to_convert=None):
    
    if columns_to_convert is None:
        columns_to_convert = [col for col in df.columns if col.endswith(suffix) and col != total_col]

    for col in columns_to_convert:
        perc_col = col.replace(suffix, new_suffix)
        df[perc_col] = (df[col] / df[total_col]) * 100

    return df

In [77]:
# Create Dataframes
student_by_age_df = pd.read_csv(student_acc_by_age_csv, skiprows = 8, nrows = 34753, skip_blank_lines = True)
student_by_age_4_15_df = pd.read_csv(student_acc_by_age_csv, skiprows = 34784, nrows = 34753, skip_blank_lines = True)
student_by_age_16_17_df = pd.read_csv(student_acc_by_age_csv, skiprows = 69560, nrows = 34753, skip_blank_lines = True)
student_by_age_18_20_df = pd.read_csv(student_acc_by_age_csv, skiprows = 104336, nrows = 34753, skip_blank_lines = True)
student_by_age_20_24_df = pd.read_csv(student_acc_by_age_csv, skiprows = 139112, nrows = 34753, skip_blank_lines = True)
student_by_age_25_above_df = pd.read_csv(student_acc_by_age_csv, skiprows = 173888, nrows = 34753, skip_blank_lines = True)

df_list = [
    student_by_age_df,
    student_by_age_4_15_df,
    student_by_age_16_17_df,
    student_by_age_18_20_df,
    student_by_age_20_24_df,
    student_by_age_25_above_df
]

student_by_age_df.head()

,2011 super output area - lower layer,All categories: Student accommodation,Living with parents,Living in a communal establishment: Total,Living in a communal establishment: University (for example halls of residence),Living in a communal establishment: Other,Living in all student household,Student living alone,Living in other household type
0,E01000001 : City of London 001A,130,89,0,0,0,10,13,18
1,E01000002 : City of London 001B,146,111,0,0,0,4,10,21
2,E01000003 : City of London 001C,188,78,65,0,65,15,16,14
3,E01000005 : City of London 001E,211,166,0,0,0,18,8,19
4,E01000006 : Barking and Dagenham 016A,421,328,0,0,0,20,5,68


In [78]:
# Dictionary for renaming columns with corrected keys
column_rename_map1 = {
    "all_categories:_student_accommodation_count": "total_students_count",
    "living_with_parents_count": "living_with_parents_count",
    "living_in_a_communal_establishment:_university_(for_example_halls_of_residence)_count": "living_in_communal_establishment_uni_count",
    "living_in_a_communal_establishment:_other_count": "living_in_communal_establishment_other_count",    
    "living_in_all_student_household_count": "living_in_all_student_household_count",
    "student_living_alone_count": "living_alone_count",
    "living_in_other_household_type_count": "living_in_other_household_count",
}

column_rename_map2 = {
    "all_categories:_student_accommodation_count": "total_students_age_4_15_count",
    "living_with_parents_count": "living_with_parents_age_4_15_count",
    "living_in_a_communal_establishment:_university_(for_example_halls_of_residence)_count": "living_in_communal_establishment_uni_age_4_15_count",
    "living_in_a_communal_establishment:_other_count": "living_in_communal_establishment_other_age_4_15_count",    
    "living_in_all_student_household_count": "living_in_all_student_household_age_4_15_count",
    "student_living_alone_count": "living_alone_age_4_15_count",
    "living_in_other_household_type_count": "living_in_other_household_age_4_15_count",
}

column_rename_map3 = {
    "all_categories:_student_accommodation_count": "total_students_age_16_17_count",
    "living_with_parents_count": "living_with_parents_age_16_17_count",
    "living_in_a_communal_establishment:_university_(for_example_halls_of_residence)_count": "living_in_communal_establishment_uni_age_16_17_count",
    "living_in_a_communal_establishment:_other_count": "living_in_communal_establishment_other_age_16_17_count",    
    "living_in_all_student_household_count": "living_in_all_student_household_age_16_17_count",
    "student_living_alone_count": "living_alone_age_16_17_count",
    "living_in_other_household_type_count": "living_in_other_household_age_16_17_count",
}

column_rename_map4 = {
    "all_categories:_student_accommodation_count": "total_students_age_18_19_count",
    "living_with_parents_count": "living_with_parents_age_18_19_count",
    "living_in_a_communal_establishment:_university_(for_example_halls_of_residence)_count": "living_in_communal_establishment_uni_age_18_19_count",
    "living_in_a_communal_establishment:_other_count": "living_in_communal_establishment_other_age_18_19_count",  
    "living_in_all_student_household_count": "living_in_all_student_household_age_18_19_count",
    "student_living_alone_count": "living_alone_age_18_19_count",
    "living_in_other_household_type_count": "living_in_other_household_age_18_19_count",
}

column_rename_map5 = {
    "all_categories:_student_accommodation_count": "total_students_age_20_24_count",
    "living_with_parents_count": "living_with_parents_age_20_24_count",
    "living_in_a_communal_establishment:_university_(for_example_halls_of_residence)_count": "living_in_communal_establishment_uni_age_20_24_count",
    "living_in_a_communal_establishment:_other_count": "living_in_communal_establishment_other_age_20_24_count", 
    "living_in_all_student_household_count": "living_in_all_student_household_age_20_24_count",
    "student_living_alone_count": "living_alone_age_20_24_count",
    "living_in_other_household_type_count": "living_in_other_household_age_20_24_count",
}

column_rename_map6 = {
    "all_categories:_student_accommodation_count": "total_students_age_25_above_count",
    "living_with_parents_count": "living_with_parents_age_25_above_count",
    "living_in_a_communal_establishment:_university_(for_example_halls_of_residence)_count": "living_in_communal_establishment_uni_age_25_above_count",
    "living_in_a_communal_establishment:_other_count": "living_in_communal_establishment_other_age_25_above_count",
    "living_in_all_student_household_count": "living_in_all_student_household_age_25_above_count",
    "student_living_alone_count": "living_alone_age_25_above_count",
    "living_in_other_household_type_count": "living_in_other_household_age_25_above_count",
}


rename_list = [column_rename_map1,
               column_rename_map2,
               column_rename_map3,
               column_rename_map4,
               column_rename_map5,
               column_rename_map6,
]

totals_list = ["total_students_count",
               "total_students_age_4_15_count",
               "total_students_age_16_17_count",
               "total_students_age_18_19_count",
               "total_students_age_20_24_count",
               "total_students_age_25_above_count",
]             
               
               
            

In [79]:
for i, df in enumerate(df_list):
    # Split the lsoa column into two new columns
    df[['lsoa11cd', 'lsoa11nm']] = df.iloc[:, 0].str.split(' : ', expand=True)
    
    # Remove the column '2021 super output area - lower layer'
    df.drop(['2011 super output area - lower layer'], 1,  inplace=True)
    
    # Rename columns 
    cols_to_rename = df.columns.difference(['lsoa11cd', 'lsoa11nm'])
    df.rename(columns={col: col.lower().replace(' ', '_') + '_count' for col in cols_to_rename}, inplace=True)
    df.rename(columns={col: col.lower().replace(':', '') for col in cols_to_rename}, inplace=True)
    
    # Remove the column '2021 super output area - lower layer'
    df.drop(['lsoa11nm','living_in_a_communal_establishment:_total_count'], 1,  inplace=True)
    
    # Move 'lsoa21cd' and 'lsoa21nm' to the front of the dataframe
    cols = df.columns.tolist()
    new_order = [cols[-1]] + cols[:-1]
    df_list[i] = df[new_order]   
    
    # Rename columns using the dictionary
    df_list[i].rename(columns=rename_list[i], inplace=True)

    # Add percentage column
    add_percentage_columns(df_list[i], totals_list[i], suffix='_count', new_suffix='_perc', columns_to_convert=df_list[i].columns[-7:])



C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\971976749.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop(['2011 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\971976749.py:14: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop(['lsoa11nm','living_in_a_communal_establishment:_total_count'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\971976749.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop(['2011 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\971976749.py:14: FutureWarning: In a future version of pandas all arguments of 

In [80]:
df_list[0].head()

,lsoa11cd,total_students_count,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,total_students_perc,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc
0,E01000001,130,89,0,0,10,13,18,100.0,68.461538,0.0,0.000000,7.692308,10.000000,13.846154
1,E01000002,146,111,0,0,4,10,21,100.0,76.027397,0.0,0.000000,2.739726,6.849315,14.383562
2,E01000003,188,78,0,65,15,16,14,100.0,41.489362,0.0,34.574468,7.978723,8.510638,7.446809
3,E01000005,211,166,0,0,18,8,19,100.0,78.672986,0.0,0.000000,8.530806,3.791469,9.004739
4,E01000006,421,328,0,0,20,5,68,100.0,77.909739,0.0,0.000000,4.750594,1.187648,16.152019


In [81]:
from functools import reduce

# Merge all DataFrames in df_list on 'lsoa21cd', keeping the order from the first df
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='lsoa11cd', how='left'),
    df_list
)

cols_to_drop = [
               "total_students_perc",
               "total_students_age_4_15_perc",
               "total_students_age_16_17_perc",
               "total_students_age_18_19_perc",
               "total_students_age_20_24_perc",
               "total_students_age_25_above_perc",
]   

merged_df.drop(cols_to_drop,1,inplace = True)

merged_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\272398875.py:18: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  merged_df.drop(cols_to_drop,1,inplace = True)


,lsoa11cd,total_students_count,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,total_students_age_4_15_count,living_with_parents_age_4_15_count,living_in_communal_establishment_uni_age_4_15_count,living_in_communal_establishment_other_age_4_15_count,living_in_all_student_household_age_4_15_count,living_alone_age_4_15_count,living_in_other_household_age_4_15_count,living_with_parents_age_4_15_perc,living_in_communal_establishment_uni_age_4_15_perc,living_in_communal_establishment_other_age_4_15_perc,living_in_all_student_household_age_4_15_perc,living_alone_age_4_15_perc,living_in_other_household_age_4_15_perc,total_students_age_16_17_count,living_with_parents_age_16_17_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_other_age_16_17_count,living_in_all_student_household_age_16_17_count,living_alone_age_16_17_count,living_in_other_household_age_16_17_count,living_with_parents_age_16_17_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_all_student_household_age_16_17_perc,living_alone_age_16_17_perc,living_in_other_household_age_16_17_perc,total_students_age_18_19_count,living_with_parents_age_18_19_count,living_in_communal_establishment_uni_age_18_19_count,living_in_communal_establishment_other_age_18_19_count,living_in_all_student_household_age_18_19_count,living_alone_age_18_19_count,living_in_other_household_age_18_19_count,living_with_parents_age_18_19_perc,living_in_communal_establishment_uni_age_18_19_perc,living_in_communal_establishment_other_age_18_19_perc,living_in_all_student_household_age_18_19_perc,living_alone_age_18_19_perc,living_in_other_household_age_18_19_perc,total_students_age_20_24_count,living_with_parents_age_20_24_count,living_in_communal_establishment_uni_age_20_24_count,living_in_communal_establishment_other_age_20_24_count,living_in_all_student_household_age_20_24_count,living_alone_age_20_24_count,living_in_other_household_age_20_24_count,living_with_parents_age_20_24_perc,living_in_communal_establishment_uni_age_20_24_perc,living_in_communal_establishment_other_age_20_24_perc,living_in_all_student_household_age_20_24_perc,living_alone_age_20_24_perc,living_in_other_household_age_20_24_perc,total_students_age_25_above_count,living_with_parents_age_25_above_count,living_in_communal_establishment_uni_age_25_above_count,living_in_communal_establishment_other_age_25_above_count,living_in_all_student_household_age_25_above_count,living_alone_age_25_above_count,living_in_other_household_age_25_above_count,living_with_parents_age_25_above_perc,living_in_communal_establishment_uni_age_25_above_perc,living_in_communal_establishment_other_age_25_above_perc,living_in_all_student_household_age_25_above_perc,living_alone_age_25_above_perc,living_in_other_household_age_25_above_perc
0,E01000001,130,89,0,0,10,13,18,68.461538,0.0,0.000000,7.692308,10.000000,13.846154,75,74,0,0,0,0,1,98.666667,0.0,0.0,0.0,0.0,1.333333,7,7,0,0,0,0,0,100.000000,0.0,0.00,0.0,0.00,0.000000,2,1,0,0,0,0,1,50.000000,0.0,0.000000,0.000000,0.000000,50.000000,29,5,0,0,10,7,7,17.241379,0.0,0.000000,34.482759,24.137931,24.137931,17,2,0,0,0,6,9,11.764706,0.0,0.000000,0.000000,35.294118,52.941176
1,E01000002,146,111,0,0,4,10,21,76.027397,0.0,0.000000,2.739726,6.849315,14.383562,86,86,0,0,0,0,0,100.000000,0.0,0.0,0.0,0.0,0.000000,11,11,0,0,0,0,0,100.000000,0.0,0.00,0.0,0.00,0.000000,10,8,0,0,0,1,1,80.000000,0.0,0.000000,0.000000,10.000000,10.000000,19,6,0,0,3,4,6,31.578947,0.0,0.000000,15.789474,21.052632,31.578947,20,0,0,0,1,5,14,0.000000,0.0,0.000000,5.000000,25.000000,70.000000
2,E010000

In [83]:
for col in totals_list:
    if col != "total_students_count":
        perc_col = col.replace('_count', '_perc')
        merged_df[perc_col] = (merged_df[col] / merged_df["total_students_count"]) * 100

merged_df.head()

,lsoa11cd,total_students_count,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,total_students_age_4_15_count,living_with_parents_age_4_15_count,living_in_communal_establishment_uni_age_4_15_count,living_in_communal_establishment_other_age_4_15_count,living_in_all_student_household_age_4_15_count,living_alone_age_4_15_count,living_in_other_household_age_4_15_count,living_with_parents_age_4_15_perc,living_in_communal_establishment_uni_age_4_15_perc,living_in_communal_establishment_other_age_4_15_perc,living_in_all_student_household_age_4_15_perc,living_alone_age_4_15_perc,living_in_other_household_age_4_15_perc,total_students_age_16_17_count,living_with_parents_age_16_17_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_other_age_16_17_count,living_in_all_student_household_age_16_17_count,living_alone_age_16_17_count,living_in_other_household_age_16_17_count,living_with_parents_age_16_17_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_all_student_household_age_16_17_perc,living_alone_age_16_17_perc,living_in_other_household_age_16_17_perc,total_students_age_18_19_count,living_with_parents_age_18_19_count,living_in_communal_establishment_uni_age_18_19_count,living_in_communal_establishment_other_age_18_19_count,living_in_all_student_household_age_18_19_count,living_alone_age_18_19_count,living_in_other_household_age_18_19_count,living_with_parents_age_18_19_perc,living_in_communal_establishment_uni_age_18_19_perc,living_in_communal_establishment_other_age_18_19_perc,living_in_all_student_household_age_18_19_perc,living_alone_age_18_19_perc,living_in_other_household_age_18_19_perc,total_students_age_20_24_count,living_with_parents_age_20_24_count,living_in_communal_establishment_uni_age_20_24_count,living_in_communal_establishment_other_age_20_24_count,living_in_all_student_household_age_20_24_count,living_alone_age_20_24_count,living_in_other_household_age_20_24_count,living_with_parents_age_20_24_perc,living_in_communal_establishment_uni_age_20_24_perc,living_in_communal_establishment_other_age_20_24_perc,living_in_all_student_household_age_20_24_perc,living_alone_age_20_24_perc,living_in_other_household_age_20_24_perc,total_students_age_25_above_count,living_with_parents_age_25_above_count,living_in_communal_establishment_uni_age_25_above_count,living_in_communal_establishment_other_age_25_above_count,living_in_all_student_household_age_25_above_count,living_alone_age_25_above_count,living_in_other_household_age_25_above_count,living_with_parents_age_25_above_perc,living_in_communal_establishment_uni_age_25_above_perc,living_in_communal_establishment_other_age_25_above_perc,living_in_all_student_household_age_25_above_perc,living_alone_age_25_above_perc,living_in_other_household_age_25_above_perc,total_students_age_4_15_perc,total_students_age_16_17_perc,total_students_age_18_19_perc,total_students_age_20_24_perc,total_students_age_25_above_perc
0,E01000001,130,89,0,0,10,13,18,68.461538,0.0,0.000000,7.692308,10.000000,13.846154,75,74,0,0,0,0,1,98.666667,0.0,0.0,0.0,0.0,1.333333,7,7,0,0,0,0,0,100.000000,0.0,0.00,0.0,0.00,0.000000,2,1,0,0,0,0,1,50.000000,0.0,0.000000,0.000000,0.000000,50.000000,29,5,0,0,10,7,7,17.241379,0.0,0.000000,34.482759,24.137931,24.137931,17,2,0,0,0,6,9,11.764706,0.0,0.000000,0.000000,35.294118,52.941176,57.692308,5.384615,1.538462,22.307692,13.076923
1,E01000002,146,111,0,0,4,10,21,76.027397,0.0,0.000000,2.739726,6.849315,14.383562,86,86,0,0,0,0,0,100.000000,0.0,0.0,0.0,0.0,0.000000,11,11,0,0,0,0,0,100.000000,0.0,0.00,0.0,0.00,0.000000,10,8,0,0,0,

In [86]:
student_pop_2021_df = pd.read_csv(student_pop_2021_csv)
student_pop_2021_df = student_pop_2021_df.groupby('lsoa11cd').sum(numeric_only=True).reset_index()
student_pop_2021_df.head()

,lsoa11cd,total_students_2021_count
0,E01000001,154
1,E01000002,141
2,E01000003,137
3,E01000005,250
4,E01000006,416


## 4. Load GIS shapefile 

In [87]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2011boundary_df = gpd.read_file(lsoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2011boundary_df.head()

Shapefile loaded successfully from the server.


,LSOA11CD,LSOA11NM,BNG_E,BNG_N,LONG_,LAT,Shape_Leng,GlobalID,geometry
0,E01000034,Barking and Dagenham 003A,550689,186433,0.172296,51.5567,6006.847623,2a20725d-7bab-4a96-b09b-154eb16073a5,"POLYGON ((550783.502 186822.004, 550783.084 18..."
1,E01000035,Barking and Dagenham 010A,549957,185575,0.161380,51.5491,3609.905134,47eef4c6-f44f-43c7-ba2f-bd8a4e62b5ee,"POLYGON ((550266.910 185957.709, 550269.100 18..."
2,E01000036,Barking and Dagenham 010B,550645,185231,0.171148,51.5459,4393.928508,57bf633a-6b2d-49f1-87fa-e64b1c2503b3,"POLYGON ((549843.501 185417.140, 549850.000 18..."
3,E01000037,Barking and Dagenham 003B,551195,187087,0.179870,51.5624,2905.634678,de173b53-0b6e-4886-a4b6-3b59153cfcf3,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E01000038,Barking and Dagenham 003C,550700,187149,0.172761,51.5631,2305.558903,16d8a8a9-39a1-40b8-a459-41b2f1fd0d04,"POLYGON ((550920.362 187341.138, 550921.876 18..."


## 5. GIS data management

### Remove Rename fields

In [88]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2011boundary_df = gpd.read_file(lsoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2011boundary_df.head()

Shapefile loaded successfully from the server.


,LSOA11CD,LSOA11NM,BNG_E,BNG_N,LONG_,LAT,Shape_Leng,GlobalID,geometry
0,E01000034,Barking and Dagenham 003A,550689,186433,0.172296,51.5567,6006.847623,2a20725d-7bab-4a96-b09b-154eb16073a5,"POLYGON ((550783.502 186822.004, 550783.084 18..."
1,E01000035,Barking and Dagenham 010A,549957,185575,0.161380,51.5491,3609.905134,47eef4c6-f44f-43c7-ba2f-bd8a4e62b5ee,"POLYGON ((550266.910 185957.709, 550269.100 18..."
2,E01000036,Barking and Dagenham 010B,550645,185231,0.171148,51.5459,4393.928508,57bf633a-6b2d-49f1-87fa-e64b1c2503b3,"POLYGON ((549843.501 185417.140, 549850.000 18..."
3,E01000037,Barking and Dagenham 003B,551195,187087,0.179870,51.5624,2905.634678,de173b53-0b6e-4886-a4b6-3b59153cfcf3,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E01000038,Barking and Dagenham 003C,550700,187149,0.172761,51.5631,2305.558903,16d8a8a9-39a1-40b8-a459-41b2f1fd0d04,"POLYGON ((550920.362 187341.138, 550921.876 18..."


In [91]:
lsoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)
lsoa2011boundary_df.rename(columns = {'LSOA11CD':'lsoa11cd','LSOA11NM':'lsoa11nm'}, inplace = True)
lsoa2011boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\318656547.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)


,lsoa11cd,lsoa11nm,geometry
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18..."
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18..."
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18..."
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18..."


### Combine with boundary lookup table

#### LSOA19 to LSOA21 to LAD22

In [92]:
lsoa19lookup = pd.read_csv(r"N:\Geodatabase\Raw_Data\Look up tables\LSOA_(2011)_to_LSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V3)(1).csv")
lsoa19lookup = lsoa19lookup[['LSOA11CD','LSOA21CD','LSOA21NM','LAD22CD','LAD22NM']]
lsoa19lookup.rename(columns = {'LSOA11CD':'lsoa11cd','LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm','LAD22CD':'lad22cd','LAD22NM':'lad22nm'}, inplace = True)
lsoa19lookup.head()

,lsoa11cd,lsoa21cd,lsoa21nm,lad22cd,lad22nm
0,E01031349,E01031349,Adur 001A,E07000223,Adur
1,E01031350,E01031350,Adur 001B,E07000223,Adur
2,E01031351,E01031351,Adur 001C,E07000223,Adur
3,E01031352,E01031352,Adur 001D,E07000223,Adur
4,E01031370,E01031370,Adur 001E,E07000223,Adur


In [93]:
lsoa2011boundary_df = pd.merge(lsoa2011boundary_df, lsoa19lookup, how = 'left', on = 'lsoa11cd')
lsoa2011boundary_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham


#### LAD22 to REGION22

In [94]:
# Define the file path for lad to region
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_26752\2710224882.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [95]:
lsoa2011boundary_df = pd.merge(lsoa2011boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2011boundary_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham,E12000007,London
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham,E12000007,London
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham,E12000007,London
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham,E12000007,London


### Add data management fields

In [96]:
# Add new data management fields with specified values
lsoa2011boundary_df['data_source'] = "Census 2011 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2011boundary_df['data_resolution'] = "LSOA 2011"
lsoa2011boundary_df['data_time_period'] = pd.to_datetime("2011-02-21")  # Convert to date format
lsoa2011boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2011boundary_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/


### Update Area

In [98]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2011boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2011boundary_df['area_ha'] = lsoa2011boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2011boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2011boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,51.365911
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,39.716142
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,41.868830
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,23.343335
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,21.404577


# 7. Combine Geometry and data

In [99]:
census2011_student_lsoa2011_gdb_df = pd.merge(lsoa2011boundary_df,merged_df, how = 'left', on='lsoa11cd')
census2011_student_lsoa2011_gdb_df = pd.merge(census2011_student_lsoa2011_gdb_df,student_pop_2021_df, how = 'left', on='lsoa11cd')
census2011_student_lsoa2011_gdb_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students_count,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,total_students_age_4_15_count,living_with_parents_age_4_15_count,living_in_communal_establishment_uni_age_4_15_count,living_in_communal_establishment_other_age_4_15_count,living_in_all_student_household_age_4_15_count,living_alone_age_4_15_count,living_in_other_household_age_4_15_count,living_with_parents_age_4_15_perc,living_in_communal_establishment_uni_age_4_15_perc,living_in_communal_establishment_other_age_4_15_perc,living_in_all_student_household_age_4_15_perc,living_alone_age_4_15_perc,living_in_other_household_age_4_15_perc,total_students_age_16_17_count,living_with_parents_age_16_17_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_other_age_16_17_count,living_in_all_student_household_age_16_17_count,living_alone_age_16_17_count,living_in_other_household_age_16_17_count,living_with_parents_age_16_17_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_all_student_household_age_16_17_perc,living_alone_age_16_17_perc,living_in_other_household_age_16_17_perc,total_students_age_18_19_count,living_with_parents_age_18_19_count,living_in_communal_establishment_uni_age_18_19_count,living_in_communal_establishment_other_age_18_19_count,living_in_all_student_household_age_18_19_count,living_alone_age_18_19_count,living_in_other_household_age_18_19_count,living_with_parents_age_18_19_perc,living_in_communal_establishment_uni_age_18_19_perc,living_in_communal_establishment_other_age_18_19_perc,living_in_all_student_household_age_18_19_perc,living_alone_age_18_19_perc,living_in_other_household_age_18_19_perc,total_students_age_20_24_count,living_with_parents_age_20_24_count,living_in_communal_establishment_uni_age_20_24_count,living_in_communal_establishment_other_age_20_24_count,living_in_all_student_household_age_20_24_count,living_alone_age_20_24_count,living_in_other_household_age_20_24_count,living_with_parents_age_20_24_perc,living_in_communal_establishment_uni_age_20_24_perc,living_in_communal_establishment_other_age_20_24_perc,living_in_all_student_household_age_20_24_perc,living_alone_age_20_24_perc,living_in_other_household_age_20_24_perc,total_students_age_25_above_count,living_with_parents_age_25_above_count,living_in_communal_establishment_uni_age_25_above_count,living_in_communal_establishment_other_age_25_above_count,living_in_all_student_household_age_25_above_count,living_alone_age_25_above_count,living_in_other_household_age_25_above_count,living_with_parents_age_25_above_perc,living_in_communal_establishment_uni_age_25_above_perc,living_in_communal_establishment_other_age_25_above_perc,living_in_all_student_household_age_25_above_perc,living_alone_age_25_above_perc,living_in_other_household_age_25_above_perc,total_students_age_4_15_perc,total_students_age_16_17_perc,total_students_age_18_19_perc,total_students_age_20_24_perc,total_students_age_25_above_perc,total_students_2021_count
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,51.365911,353,331,0,2,2,0,18,93.767705,0.0,0.566572,0.566572,0.000000,5.099150,248,245,0,0,0,0,3,98.790323,0.0,0.0,0.0,0.0,1.209677,51,48,0,0,0,0,3,94.117647,0.0,0.000000,0.0,0.0,5.882353,26,25,0,0,0,0,

In [100]:
census2011_student_lsoa2011_gdb_df['change_instudent_population_10_yr'] = ((census2011_student_lsoa2011_gdb_df['total_students_2021_count'] - census2011_student_lsoa2011_gdb_df['total_students_count'])/census2011_student_lsoa2011_gdb_df['total_students_count'])*100
census2011_student_lsoa2011_gdb_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students_count,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,total_students_age_4_15_count,living_with_parents_age_4_15_count,living_in_communal_establishment_uni_age_4_15_count,living_in_communal_establishment_other_age_4_15_count,living_in_all_student_household_age_4_15_count,living_alone_age_4_15_count,living_in_other_household_age_4_15_count,living_with_parents_age_4_15_perc,living_in_communal_establishment_uni_age_4_15_perc,living_in_communal_establishment_other_age_4_15_perc,living_in_all_student_household_age_4_15_perc,living_alone_age_4_15_perc,living_in_other_household_age_4_15_perc,total_students_age_16_17_count,living_with_parents_age_16_17_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_other_age_16_17_count,living_in_all_student_household_age_16_17_count,living_alone_age_16_17_count,living_in_other_household_age_16_17_count,living_with_parents_age_16_17_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_all_student_household_age_16_17_perc,living_alone_age_16_17_perc,living_in_other_household_age_16_17_perc,total_students_age_18_19_count,living_with_parents_age_18_19_count,living_in_communal_establishment_uni_age_18_19_count,living_in_communal_establishment_other_age_18_19_count,living_in_all_student_household_age_18_19_count,living_alone_age_18_19_count,living_in_other_household_age_18_19_count,living_with_parents_age_18_19_perc,living_in_communal_establishment_uni_age_18_19_perc,living_in_communal_establishment_other_age_18_19_perc,living_in_all_student_household_age_18_19_perc,living_alone_age_18_19_perc,living_in_other_household_age_18_19_perc,total_students_age_20_24_count,living_with_parents_age_20_24_count,living_in_communal_establishment_uni_age_20_24_count,living_in_communal_establishment_other_age_20_24_count,living_in_all_student_household_age_20_24_count,living_alone_age_20_24_count,living_in_other_household_age_20_24_count,living_with_parents_age_20_24_perc,living_in_communal_establishment_uni_age_20_24_perc,living_in_communal_establishment_other_age_20_24_perc,living_in_all_student_household_age_20_24_perc,living_alone_age_20_24_perc,living_in_other_household_age_20_24_perc,total_students_age_25_above_count,living_with_parents_age_25_above_count,living_in_communal_establishment_uni_age_25_above_count,living_in_communal_establishment_other_age_25_above_count,living_in_all_student_household_age_25_above_count,living_alone_age_25_above_count,living_in_other_household_age_25_above_count,living_with_parents_age_25_above_perc,living_in_communal_establishment_uni_age_25_above_perc,living_in_communal_establishment_other_age_25_above_perc,living_in_all_student_household_age_25_above_perc,living_alone_age_25_above_perc,living_in_other_household_age_25_above_perc,total_students_age_4_15_perc,total_students_age_16_17_perc,total_students_age_18_19_perc,total_students_age_20_24_perc,total_students_age_25_above_perc,total_students_2021_count,change_instudent_population_10_yr
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,51.365911,353,331,0,2,2,0,18,93.767705,0.0,0.566572,0.566572,0.000000,5.099150,248,245,0,0,0,0,3,98.790323,0.0,0.0,0.0,0.0,1.209677,51,48,0,0,0,0,3,94.117647,0.0,0.0000

# 8. Final tweaks

In [102]:
# Reorder columns in the combined dataframe

final_column_order = ['lsoa11cd',
 'lsoa11nm',
 'geometry',
 'lsoa21cd',
 'lsoa21nm',
 'lad22cd',
 'lad22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'total_students_count',
 'total_students_age_4_15_count',
 'total_students_age_16_17_count',
 'total_students_age_18_19_count',
 'total_students_age_20_24_count',
 'total_students_age_25_above_count',
                      
 'total_students_age_4_15_perc',
 'total_students_age_16_17_perc',
 'total_students_age_18_19_perc',
 'total_students_age_20_24_perc',
 'total_students_age_25_above_perc',                      
                      
 'living_with_parents_count',
 'living_in_communal_establishment_uni_count',
 'living_in_communal_establishment_other_count',
 'living_in_all_student_household_count',
 'living_alone_count',
 'living_in_other_household_count',
 'living_with_parents_perc',
 'living_in_communal_establishment_uni_perc',
 'living_in_communal_establishment_other_perc',
 'living_in_all_student_household_perc',
 'living_alone_perc',
 'living_in_other_household_perc',
 
 'living_with_parents_age_4_15_count',
 'living_in_communal_establishment_uni_age_4_15_count',
 'living_in_communal_establishment_other_age_4_15_count',
 'living_in_all_student_household_age_4_15_count',
 'living_alone_age_4_15_count',
 'living_in_other_household_age_4_15_count',
 'living_with_parents_age_4_15_perc',
 'living_in_communal_establishment_uni_age_4_15_perc',
 'living_in_communal_establishment_other_age_4_15_perc',
 'living_in_all_student_household_age_4_15_perc',
 'living_alone_age_4_15_perc',
 'living_in_other_household_age_4_15_perc',
 
 'living_with_parents_age_16_17_count',
 'living_in_communal_establishment_uni_age_16_17_count',
 'living_in_communal_establishment_other_age_16_17_count',
 'living_in_all_student_household_age_16_17_count',
 'living_alone_age_16_17_count',
 'living_in_other_household_age_16_17_count',
 'living_with_parents_age_16_17_perc',
 'living_in_communal_establishment_uni_age_16_17_perc',
 'living_in_communal_establishment_other_age_16_17_perc',
 'living_in_all_student_household_age_16_17_perc',
 'living_alone_age_16_17_perc',
 'living_in_other_household_age_16_17_perc',
 
 'living_with_parents_age_18_19_count',
 'living_in_communal_establishment_uni_age_18_19_count',
 'living_in_communal_establishment_other_age_18_19_count',
 'living_in_all_student_household_age_18_19_count',
 'living_alone_age_18_19_count',
 'living_in_other_household_age_18_19_count',
 'living_with_parents_age_18_19_perc',
 'living_in_communal_establishment_uni_age_18_19_perc',
 'living_in_communal_establishment_other_age_18_19_perc',
 'living_in_all_student_household_age_18_19_perc',
 'living_alone_age_18_19_perc',
 'living_in_other_household_age_18_19_perc',

 'living_with_parents_age_20_24_count',
 'living_in_communal_establishment_uni_age_20_24_count',
 'living_in_communal_establishment_other_age_20_24_count',
 'living_in_all_student_household_age_20_24_count',
 'living_alone_age_20_24_count',
 'living_in_other_household_age_20_24_count',
 'living_with_parents_age_20_24_perc',
 'living_in_communal_establishment_uni_age_20_24_perc',
 'living_in_communal_establishment_other_age_20_24_perc',
 'living_in_all_student_household_age_20_24_perc',
 'living_alone_age_20_24_perc',
 'living_in_other_household_age_20_24_perc',
 
 'living_with_parents_age_25_above_count',
 'living_in_communal_establishment_uni_age_25_above_count',
 'living_in_communal_establishment_other_age_25_above_count',
 'living_in_all_student_household_age_25_above_count',
 'living_alone_age_25_above_count',
 'living_in_other_household_age_25_above_count',
 'living_with_parents_age_25_above_perc',
 'living_in_communal_establishment_uni_age_25_above_perc',
 'living_in_communal_establishment_other_age_25_above_perc',
 'living_in_all_student_household_age_25_above_perc',
 'living_alone_age_25_above_perc',
 'living_in_other_household_age_25_above_perc',
 
 'total_students_2021_count',
 'change_instudent_population_10_yr']

census2011_student_lsoa2011_gdb_df = census2011_student_lsoa2011_gdb_df[final_column_order]

census2011_student_lsoa2011_gdb_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students_count,total_students_age_4_15_count,total_students_age_16_17_count,total_students_age_18_19_count,total_students_age_20_24_count,total_students_age_25_above_count,total_students_age_4_15_perc,total_students_age_16_17_perc,total_students_age_18_19_perc,total_students_age_20_24_perc,total_students_age_25_above_perc,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,living_with_parents_age_4_15_count,living_in_communal_establishment_uni_age_4_15_count,living_in_communal_establishment_other_age_4_15_count,living_in_all_student_household_age_4_15_count,living_alone_age_4_15_count,living_in_other_household_age_4_15_count,living_with_parents_age_4_15_perc,living_in_communal_establishment_uni_age_4_15_perc,living_in_communal_establishment_other_age_4_15_perc,living_in_all_student_household_age_4_15_perc,living_alone_age_4_15_perc,living_in_other_household_age_4_15_perc,living_with_parents_age_16_17_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_other_age_16_17_count,living_in_all_student_household_age_16_17_count,living_alone_age_16_17_count,living_in_other_household_age_16_17_count,living_with_parents_age_16_17_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_all_student_household_age_16_17_perc,living_alone_age_16_17_perc,living_in_other_household_age_16_17_perc,living_with_parents_age_18_19_count,living_in_communal_establishment_uni_age_18_19_count,living_in_communal_establishment_other_age_18_19_count,living_in_all_student_household_age_18_19_count,living_alone_age_18_19_count,living_in_other_household_age_18_19_count,living_with_parents_age_18_19_perc,living_in_communal_establishment_uni_age_18_19_perc,living_in_communal_establishment_other_age_18_19_perc,living_in_all_student_household_age_18_19_perc,living_alone_age_18_19_perc,living_in_other_household_age_18_19_perc,living_with_parents_age_20_24_count,living_in_communal_establishment_uni_age_20_24_count,living_in_communal_establishment_other_age_20_24_count,living_in_all_student_household_age_20_24_count,living_alone_age_20_24_count,living_in_other_household_age_20_24_count,living_with_parents_age_20_24_perc,living_in_communal_establishment_uni_age_20_24_perc,living_in_communal_establishment_other_age_20_24_perc,living_in_all_student_household_age_20_24_perc,living_alone_age_20_24_perc,living_in_other_household_age_20_24_perc,living_with_parents_age_25_above_count,living_in_communal_establishment_uni_age_25_above_count,living_in_communal_establishment_other_age_25_above_count,living_in_all_student_household_age_25_above_count,living_alone_age_25_above_count,living_in_other_household_age_25_above_count,living_with_parents_age_25_above_perc,living_in_communal_establishment_uni_age_25_above_perc,living_in_communal_establishment_other_age_25_above_perc,living_in_all_student_household_age_25_above_perc,living_alone_age_25_above_perc,living_in_other_household_age_25_above_perc,total_students_2021_count,change_instudent_population_10_yr
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,51.365911,353,248,51,26,15,13,70.254958,14.447592,7.365439,4.249292,3.682720,331,0,2,2,0,18,93.767705,0.0,0.566572,0.566572,0.000000,5.099150,245,0,0,0,0,3,98.790323,0.

# Special Case: Find and remove Duplicates

In [104]:
def remove_duplicates(df, unique_column):
    """
    Removes duplicate rows based on a specified unique column while keeping the first occurrence.
    
    Parameters:
        df (GeoDataFrame or DataFrame): The input data to clean.
        unique_column (str): The column to check for duplicates.
    
    Returns:
        GeoDataFrame or DataFrame: A cleaned dataframe with duplicates removed.
    """
    # Check for duplicates
    duplicate_count = df.duplicated(subset=[unique_column], keep='first').sum()
    
    if duplicate_count > 0:
        print(f"Found {duplicate_count} duplicate(s) in column '{unique_column}'. Removing duplicates...")

        # Remove duplicates, keeping the first occurrence
        df = df.drop_duplicates(subset=[unique_column], keep='first')

        print(f"Duplicates removed. Now {unique_column} has only unique values.")

    else:
        print(f"No duplicates found in column '{unique_column}'. No changes made.")

    return df

# Specify the column where duplicates should be checked
unique_column = "lsoa11cd"

# Remove duplicates before uploading to PostGIS
census2011_student_lsoa2011_gdb_df = remove_duplicates(census2011_student_lsoa2011_gdb_df, unique_column)

census2011_student_lsoa2011_gdb_df.head()

No duplicates found in column 'lsoa11cd'. No changes made.


,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students_count,total_students_age_4_15_count,total_students_age_16_17_count,total_students_age_18_19_count,total_students_age_20_24_count,total_students_age_25_above_count,total_students_age_4_15_perc,total_students_age_16_17_perc,total_students_age_18_19_perc,total_students_age_20_24_perc,total_students_age_25_above_perc,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,living_with_parents_age_4_15_count,living_in_communal_establishment_uni_age_4_15_count,living_in_communal_establishment_other_age_4_15_count,living_in_all_student_household_age_4_15_count,living_alone_age_4_15_count,living_in_other_household_age_4_15_count,living_with_parents_age_4_15_perc,living_in_communal_establishment_uni_age_4_15_perc,living_in_communal_establishment_other_age_4_15_perc,living_in_all_student_household_age_4_15_perc,living_alone_age_4_15_perc,living_in_other_household_age_4_15_perc,living_with_parents_age_16_17_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_other_age_16_17_count,living_in_all_student_household_age_16_17_count,living_alone_age_16_17_count,living_in_other_household_age_16_17_count,living_with_parents_age_16_17_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_all_student_household_age_16_17_perc,living_alone_age_16_17_perc,living_in_other_household_age_16_17_perc,living_with_parents_age_18_19_count,living_in_communal_establishment_uni_age_18_19_count,living_in_communal_establishment_other_age_18_19_count,living_in_all_student_household_age_18_19_count,living_alone_age_18_19_count,living_in_other_household_age_18_19_count,living_with_parents_age_18_19_perc,living_in_communal_establishment_uni_age_18_19_perc,living_in_communal_establishment_other_age_18_19_perc,living_in_all_student_household_age_18_19_perc,living_alone_age_18_19_perc,living_in_other_household_age_18_19_perc,living_with_parents_age_20_24_count,living_in_communal_establishment_uni_age_20_24_count,living_in_communal_establishment_other_age_20_24_count,living_in_all_student_household_age_20_24_count,living_alone_age_20_24_count,living_in_other_household_age_20_24_count,living_with_parents_age_20_24_perc,living_in_communal_establishment_uni_age_20_24_perc,living_in_communal_establishment_other_age_20_24_perc,living_in_all_student_household_age_20_24_perc,living_alone_age_20_24_perc,living_in_other_household_age_20_24_perc,living_with_parents_age_25_above_count,living_in_communal_establishment_uni_age_25_above_count,living_in_communal_establishment_other_age_25_above_count,living_in_all_student_household_age_25_above_count,living_alone_age_25_above_count,living_in_other_household_age_25_above_count,living_with_parents_age_25_above_perc,living_in_communal_establishment_uni_age_25_above_perc,living_in_communal_establishment_other_age_25_above_perc,living_in_all_student_household_age_25_above_perc,living_alone_age_25_above_perc,living_in_other_household_age_25_above_perc,total_students_2021_count,change_instudent_population_10_yr
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,Census 2011 (ONS Nomis - Official Census and L...,LSOA 2011,2011-02-21,https://www.nomisweb.co.uk/,51.365911,353,248,51,26,15,13,70.254958,14.447592,7.365439,4.249292,3.682720,331,0,2,2,0,18,93.767705,0.0,0.566572,0.566572,0.000000,5.099150,245,0,0,0,0,3,98.790323,0.

# 9. Upload to geodatabase

In [105]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2011_lsoa2011_students"  # Desired table name
primary_key_column = "lsoa11cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [106]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2011_student_lsoa2011_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2011_student_lsoa2011_gdb_df.set_crs(epsg=27700, inplace=True)

In [107]:
# Publish the GeoDataFrame to PostGIS
census2011_student_lsoa2011_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2011_lsoa2011_students
Primary key set on column: lsoa11cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2011_lsoa2011_students
